# Multi-Player Prop Prediction Pipeline
This notebook loads the latest starting XIs for two national teams, fetches their latest baseline and player ELO ratings, applies the new formation matchup features, and runs batch predictions across multiple targets at once.

In [ ]:
import sys
import importlib
from pathlib import Path

# Set repository root path (change this if running on a remote Databricks workspace path)
repo = "/Workspace/Users/adam.r.denes@gmail.com/football-analytics"
if repo not in sys.path:
    sys.path.insert(0, repo)
    sys.path.insert(0, f"{repo}/scripts")

In [ ]:
import pandas as pd
import numpy as np
import math
import mlflow
from databricks.sdk import WorkspaceClient

# Configure MLflow to bypass Spark Connect config checks and talk directly to Unity Catalog
mlflow.set_registry_uri("databricks-uc")

# Match Config
team_a_name = "Brazil"
team_b_name = "Norway"
projected_minutes = 90.0

team_a_formation = "4-3-3"
team_b_formation = "4-2-3-1"

feature_table = "football_analytics.gold.fct_football__player_shot_features"
team_elo_table = "football_analytics.gold.fct_football__team_elo_history"
matchup_table = "football_analytics.gold.fct_football__formation_matchup_history"

print("Loading gold ELO and features logs...")
history = spark.table(feature_table).toPandas()
team_history = spark.table(team_elo_table).toPandas()

# Look up the latest pre-match ELO for both teams
elo_a = team_history[team_history["team_name"] == team_a_name].sort_values("fixture_date_utc").tail(1)
elo_b = team_history[team_history["team_name"] == team_b_name].sort_values("fixture_date_utc").tail(1)

if elo_a.empty or elo_b.empty:
    raise ValueError(f"Could not find ELO history for {team_a_name} or {team_b_name}")

elo_a_gen = elo_a["team_elo_general_pre"].values[0]
elo_a_att = elo_a["team_elo_attack_pre"].values[0]
elo_a_def = elo_a["team_elo_defense_pre"].values[0]

elo_b_gen = elo_b["team_elo_general_pre"].values[0]
elo_b_att = elo_b["team_elo_attack_pre"].values[0]
elo_b_def = elo_b["team_elo_defense_pre"].values[0]

# Auto-detect the latest recorded starting lineup for each team
def get_latest_starting_xi(team_name):
    team_fixtures = history[history["team_name"] == team_name]
    if team_fixtures.empty:
        raise ValueError(f"No player caps found for team: {team_name}")
    latest_fixture = team_fixtures.sort_values("fixture_date_utc").tail(1)["fixture_id"].values[0]
    return team_fixtures[
        (team_fixtures["fixture_id"] == latest_fixture) & 
        (team_fixtures["is_starter"] == True)
    ]

starters_a = get_latest_starting_xi(team_a_name)
starters_b = get_latest_starting_xi(team_b_name)

print(f"Loaded {len(starters_a)} starters for {team_a_name}")
print(f"Loaded {len(starters_b)} starters for {team_b_name}")

In [ ]:
# Build inference context dataset
inference_rows = []

# Process Team A starters playing against Team B
for _, player in starters_a.iterrows():
    p = player.copy()
    p["opponent_team_name"] = team_b_name
    p["opponent_elo_general_pre"] = elo_b_gen
    p["opponent_elo_attack_pre"] = elo_b_att
    p["opponent_elo_defense_pre"] = elo_b_def
    p["formation"] = team_a_formation
    p["opponent_formation"] = team_b_formation
    
    # Set default/baseline values for formation matchups (lookups can be customized)
    p["formation_win_rate_pre"] = 0.5
    p["formation_count_pre"] = 0
    p["formation_matchup_win_rate_pre"] = 0.5
    p["formation_matchup_count_pre"] = 0
    
    p["games_minutes"] = projected_minutes
    p["exposure"] = projected_minutes / 90.0
    p["is_starter"] = True
    p["was_substitute"] = False
    inference_rows.append(p)

# Process Team B starters playing against Team A
for _, player in starters_b.iterrows():
    p = player.copy()
    p["opponent_team_name"] = team_a_name
    p["opponent_elo_general_pre"] = elo_a_gen
    p["opponent_elo_attack_pre"] = elo_a_att
    p["opponent_elo_defense_pre"] = elo_a_def
    p["formation"] = team_b_formation
    p["opponent_formation"] = team_a_formation
    
    p["formation_win_rate_pre"] = 0.5
    p["formation_count_pre"] = 0
    p["formation_matchup_win_rate_pre"] = 0.5
    p["formation_matchup_count_pre"] = 0
    
    p["games_minutes"] = projected_minutes
    p["exposure"] = projected_minutes / 90.0
    p["is_starter"] = True
    p["was_substitute"] = False
    inference_rows.append(p)

df_inference = pd.DataFrame(inference_rows)

# Calculate derived interactive ELO delta features
df_inference["team_elo_general_diff"] = df_inference["team_elo_general_pre"] - df_inference["opponent_elo_general_pre"]
df_inference["team_attack_vs_opp_defense"] = df_inference["team_elo_attack_pre"] - df_inference["opponent_elo_defense_pre"]
df_inference["team_defense_vs_opp_attack"] = df_inference["team_elo_defense_pre"] - df_inference["opponent_elo_attack_pre"]
df_inference["player_attack_vs_opp_defense"] = df_inference["player_offensive_elo_pre"] - df_inference["opponent_elo_defense_pre"]
df_inference["player_defense_vs_opp_attack"] = df_inference["player_defensive_elo_pre"] - df_inference["opponent_elo_attack_pre"]
df_inference["lineup_attack_vs_opp_defense"] = df_inference["team_lineup_attack_strength"] - df_inference["opponent_elo_defense_pre"]
df_inference["lineup_defense_vs_opp_attack"] = df_inference["team_lineup_defense_strength"] - df_inference["opponent_elo_attack_pre"]
df_inference["player_attack_delta_vs_team"] = df_inference["player_offensive_elo_pre"] - df_inference["team_elo_attack_pre"]
df_inference["player_defense_delta_vs_team"] = df_inference["player_defensive_elo_pre"] - df_inference["team_elo_defense_pre"]
df_inference["lineup_attack_delta_vs_team"] = df_inference["team_lineup_attack_strength"] - df_inference["team_elo_attack_pre"]
df_inference["lineup_defense_delta_vs_team"] = df_inference["team_lineup_defense_strength"] - df_inference["team_elo_defense_pre"]

# Ensure types match model expectation
df_inference["games_minutes"] = df_inference["games_minutes"].astype(float)
df_inference["formation_row"] = df_inference["formation_row"].astype(float)
df_inference["formation_column"] = df_inference["formation_column"].astype(float)

In [ ]:
w = WorkspaceClient()

targets = ["shots_total", "shots_on", "goals_total", "fouls_committed", "dribbles_attempts"]
model_prefix = "football_analytics.gold.player_prop_poisson_lgbm"

# Ensure MLflow registry is explicitly set for Unity Catalog models
mlflow.set_registry_uri("databricks-uc")

# Store predictions per target
target_predictions = {}

for target in targets:
    model_name = f"{model_prefix}_{target}"
    
    versions = list(w.model_versions.list(model_name))
    if not versions:
        print(f"Skipping target: {target} (no model registered)")
        continue
    latest_version = max([v.version for v in versions])
    
    model_uri = f"models:/{model_name}/{latest_version}"
    model = mlflow.pyfunc.load_model(model_uri)
    
    # Batch predict
    predicted_rates = model.predict(df_inference)
    target_predictions[target] = [rate * (projected_minutes / 90.0) for rate in predicted_rates]

# Build output summary table
summary_rows = []
for idx, (_, row) in enumerate(df_inference.iterrows()):
    record = {
        "Player": row["player_name"],
        "Team": row["team_name"],
        "Pos": row["primary_position"]
    }
    for target in targets:
        if target in target_predictions:
            record[f"Exp {target}"] = round(target_predictions[target][idx], 3)
    summary_rows.append(record)

df_results = pd.DataFrame(summary_rows)
display(df_results)